# ⚛️ Módulo 3: Mecánica Molecular y Campos de Fuerza
## Actividad 3.7: Software Especializado

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_03_mecanica_molecular/07_software_especializado.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Distinguir los principales paquetes de software para mecánica molecular
- Usar RDKit para manipulación avanzada de moléculas y propiedades
- Emplear OpenBabel para conversión de formatos y operaciones básicas
- Visualizar moléculas en 3D dentro del notebook con Py3Dmol
- Diseñar flujos de trabajo que combinen múltiples herramientas
- Seleccionar la herramienta adecuada para cada tarea

---

## 📚 Introducción

El ecosistema de software para química computacional es vasto y especializado. Cada herramienta tiene su nicho y sus fortalezas. En esta actividad exploraremos las herramientas más relevantes para mecánica molecular con acceso libre y compatibles con Python.

### Ecosistema de Software MM

| Software | Tipo | Fortaleza Principal | Acceso |
|----------|------|--------------------|---------|
| **RDKit** | Librería Python | Quiminformática, descriptores, conformeros | Libre |
| **OpenBabel** | CLI + Python | Conversión de formatos, SMILES, operaciones | Libre |
| **Py3Dmol** | Visualización | Visualización 3D en Jupyter | Libre |
| **GROMACS** | Simulación | Dinámica molecular de alto rendimiento | Libre |
| **AMBER** | Simulación | MD + campos de fuerza biomoleculares | Académico |
| **NAMD** | Simulación | MD en GPU, escalable | Libre |
| **Tinker** | MM educacional | Fácil de usar, muchos campos de fuerza | Libre |
| **Schrodinger** | Suite completa | Docking, MM/QMMM, ADMET | Comercial |
| **CHARMM** | MM/QM/MD | Biomoléculas, parámetros CHARMM FF | Académico |

### Criterios de Selección

La elección de software depende de:
- **Escala**: pequeñas moléculas vs proteínas vs sistemas periódicos
- **Campo de fuerza**: MMFF94, UFF, AMBER, CHARMM, OPLS...
- **Integración**: ¿con Python? ¿con flujos automatizados?
- **Recursos**: CPU/GPU, memoria, tiempo de cómputo

In [ ]:
!pip install rdkit-pypi numpy scipy matplotlib 2>/dev/null || \
  pip install rdkit numpy scipy matplotlib
!pip install py3Dmol 2>/dev/null || echo 'Instalar manualmente: pip install py3Dmol'
print('✓ Instalación completada')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import subprocess
import os
import warnings
warnings.filterwarnings('ignore')

# RDKit
try:
    from rdkit import Chem, __version__ as rdkit_ver
    from rdkit.Chem import AllChem, Draw, Descriptors
    from rdkit.Chem import rdMolDescriptors, rdDistGeom
    RDKIT_OK = True
    print(f'✓ RDKit {rdkit_ver}')
except ImportError:
    RDKIT_OK = False
    print('⚠️  RDKit no disponible')

# Py3Dmol
try:
    import py3Dmol
    PY3DMOL_OK = True
    print('✓ Py3Dmol disponible')
except ImportError:
    PY3DMOL_OK = False
    print('⚠️  Py3Dmol no disponible (instalar con: pip install py3Dmol)')

# OpenBabel
try:
    result = subprocess.run(['obabel', '--version'],
                           capture_output=True, text=True, timeout=5)
    OBABEL_OK = result.returncode == 0
    if OBABEL_OK:
        print(f'✓ OpenBabel disponible')
    else:
        print('⚠️  OpenBabel no encontrado en PATH')
except (FileNotFoundError, subprocess.TimeoutExpired):
    OBABEL_OK = False
    print('⚠️  OpenBabel no disponible (instalar con: sudo apt install openbabel)')

print('\n📦 Estado del software:')
for nom, ok in [('RDKit', RDKIT_OK), ('Py3Dmol', PY3DMOL_OK), ('OpenBabel', OBABEL_OK)]:
    estado = '✓ Disponible' if ok else '✗ No disponible'
    print(f'  {nom:12s}: {estado}')

## 1. RDKit — Manipulación Avanzada de Moléculas

In [ ]:
def demostrar_rdkit(smiles_lista):
    """
    Demuestra las capacidades de RDKit para manipulación molecular.
    """
    if not RDKIT_OK:
        print('RDKit no disponible. Mostrando ejemplos teóricos...')
        print('Capacidades de RDKit:')
        print('  1. SMILES → Molécula 3D: EmbedMolecule()')
        print('  2. Optimización: MMFFOptimizeMolecule()')
        print('  3. Descriptores: Descriptors.MolWt(), MolLogP(), TPSA()')
        print('  4. Búsqueda de subestructuras: HasSubstructMatch()')
        print('  5. Conversión de formatos: MolToMolBlock(), MolToPDBBlock()')
        return

    print('=== RDKIT: CAPACIDADES PRINCIPALES ===')
    resultados = []

    for smi in smiles_lista:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        nombre = mol.GetProp('_Name') if mol.HasProp('_Name') else smi[:20]

        # 1. Información básica
        n_atomos = mol.GetNumAtoms()
        n_enlaces = mol.GetNumBonds()
        formula = rdMolDescriptors.CalcMolFormula(mol)

        # 2. Descriptores
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        tpsa = Descriptors.TPSA(mol)

        # 3. Generar 3D
        mol3d = Chem.AddHs(mol)
        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        AllChem.EmbedMolecule(mol3d, params)
        AllChem.MMFFOptimizeMolecule(mol3d)

        # 4. Exportar a MOL/PDB
        molblock = Chem.MolToMolBlock(mol3d)
        pdbblock = Chem.MolToPDBBlock(mol3d)

        resultados.append({
            'SMILES': smi,
            'Fórmula': formula,
            'MW': round(mw, 2),
            'LogP': round(logp, 2),
            'TPSA': round(tpsa, 1),
            '# átomos': n_atomos,
            '# enlaces': n_enlaces,
            'MOL chars': len(molblock),
            'PDB chars': len(pdbblock),
        })

        print(f'\n  {formula} ({smi[:30]})')
        print(f'    MW={mw:.1f}, LogP={logp:.2f}, TPSA={tpsa:.1f}')
        print(f'    3D generado: {mol3d.GetNumConformers()} conformero(s)')

    import pandas as pd
    return pd.DataFrame(resultados)

smiles_demo = [
    'CCO',
    'CC(=O)Oc1ccccc1C(=O)O',  # Aspirina
    'CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C',  # Testosterona
    'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',  # Cafeína
]

df_rdkit = demostrar_rdkit(smiles_demo)

## 2. Conversión de Formatos con OpenBabel

In [ ]:
def convertir_con_obabel(input_str, input_fmt, output_fmt, nombre='mol'):
    """
    Convierte entre formatos moleculares usando OpenBabel.
    input_str: contenido del archivo de entrada (o SMILES)
    """
    if not OBABEL_OK:
        print(f'OpenBabel no disponible.')
        print(f'Ejemplo de uso en terminal:')
        print(f'  obabel -i{input_fmt} input.{input_fmt} -o{output_fmt} -O output.{output_fmt}')
        print(f'  obabel -:"CCO" -ismi -omol2 -O ethanol.mol2 --gen3d')
        return None

    cmd = ['obabel', f'-i{input_fmt}', '-', f'-o{output_fmt}']
    if input_fmt == 'smi':
        cmd = ['obabel', f'-:"{input_str}"', '-ismi', f'-o{output_fmt}', '--gen3d']

    try:
        result = subprocess.run(
            cmd,
            input=input_str if input_fmt != 'smi' else None,
            capture_output=True,
            text=True,
            timeout=30
        )
        if result.returncode == 0:
            return result.stdout
        else:
            print(f'Error: {result.stderr}')
            return None
    except Exception as e:
        print(f'Error ejecutando OpenBabel: {e}')
        return None

def demostrar_obabel():
    """Demuestra las capacidades de OpenBabel."""
    print('=== OPENBABEL: CAPACIDADES PRINCIPALES ===')
    print()

    if not OBABEL_OK:
        print('🔧 OpenBabel no disponible. Ejemplos de operaciones típicas:')
        print()
        ejemplos = [
            ('Conversión SDF → PDB',
             'obabel input.sdf -o pdb -O output.pdb',
             'Cambiar formato manteniendo coordenadas 3D'),
            ('Generación 3D desde SMILES',
             'obabel -:"CCO" -ismi -omol2 --gen3d -O ethanol.mol2',
             'Generar coordenadas 3D y exportar a MOL2'),
            ('Conversión SMILES → InChI',
             'obabel -:"CC(=O)O" -ismi -oinchi',
             'Obtener identificador estándar InChI'),
            ('Minimización MM',
             'obabel input.mol -o mol -O output_min.mol --minimize --ff MMFF94',
             'Optimización geométrica con MMFF94'),
            ('Cálculo de torsiones',
             'obabel input.sdf --torsum',
             'Sumar todos los ángulos de torsión'),
            ('Estandarizar cargas',
             'obabel input.sdf -o sdf -O neutral.sdf --neutralize',
             'Neutralizar moléculas cargadas'),
            ('Cribado SMARTS',
             'obabel database.sdf -s "c1ccccc1" -O hits.sdf',
             'Buscar subestructuras con SMARTS'),
        ]

        for tit, cmd, desc in ejemplos:
            print(f'  📌 {tit}')
            print(f'     Comando: {cmd}')
            print(f'     Uso: {desc}')
            print()

        print('\n  Formatos soportados (>100):')
        formatos = ['mol', 'mol2', 'sdf', 'pdb', 'xyz', 'smi', 'inchi',
                   'cif', 'gro', 'cube', 'log (Gaussian)', 'out (ORCA)']
        for i in range(0, len(formatos), 4):
            print('    ' + '  │  '.join(f'{f:16s}' for f in formatos[i:i+4]))
        return

    # Si está disponible, hacer conversiones reales
    smi_etanol = 'CCO ethanol'
    result = subprocess.run(
        ['obabel', '-ismi', '-', '-osdf', '--gen3d'],
        input=smi_etanol, capture_output=True, text=True, timeout=30
    )
    if result.returncode == 0:
        print('✓ Etanol generado en 3D:')
        print(result.stdout[:300])

demostrar_obabel()

## 3. Visualización 3D con Py3Dmol

In [ ]:
def visualizar_mol_py3dmol(smiles, nombre, estilo='stick', color_scheme='ssPyMol'):
    """
    Visualiza una molécula en 3D con Py3Dmol.
    Requiere: pip install py3Dmol y RDKit
    """
    if not RDKIT_OK:
        print(f'{nombre}: Necesita RDKit para generar 3D')
        return

    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    AllChem.EmbedMolecule(mol, params)
    AllChem.MMFFOptimizeMolecule(mol)
    molblock = Chem.MolToMolBlock(mol)

    if not PY3DMOL_OK:
        print(f'Py3Dmol no disponible. Coordenadas de {nombre} generadas.')
        conf = mol.GetConformer()
        print(f'  Coordenadas ({mol.GetNumAtoms()} átomos):')
        for i, atom in enumerate(mol.GetAtoms()):
            pos = conf.GetAtomPosition(i)
            print(f'  {atom.GetSymbol():2s}  {pos.x:8.3f}  {pos.y:8.3f}  {pos.z:8.3f}')
        return molblock

    view = py3Dmol.view(width=600, height=400)
    view.addModel(molblock, 'mol')
    view.setStyle({estilo: {}})
    view.zoomTo()
    view.setBackgroundColor('white')
    print(f'Visualizando: {nombre}')
    return view.show()

def comparar_visualizaciones():
    """
    Compara diferentes estilos de visualización con Py3Dmol.
    """
    print('=== PY3DMOL: ESTILOS DE VISUALIZACIÓN ===')
    print()

    estilos = {
        'stick': 'Barras — muestra enlacesy ángulos claramente',
        'sphere': 'VdW Spheres — muestra tamaño y volumen molecular',
        'line': 'Wireframe — vista compacta, útil para muchas moléculas',
        'cartoon': 'Ribbon — para proteínas, muestra estructura secundaria',
        'surface': 'Superficie — muestra cavidades y sitios de unión',
    }

    print('  Estilos disponibles en Py3Dmol:')
    for est, desc in estilos.items():
        print(f'  • {est:10s}: {desc}')

    print()
    print('  Ejemplo de uso básico:')
    print('''
  import py3Dmol

  view = py3Dmol.view(width=600, height=400)
  view.addModel(molblock, "mol")       # molblock = string MOL/SDF/PDB
  view.setStyle({"stick": {}})         # estilo de representación
  view.addSurface(py3Dmol.VDW,         # superficie VdW
                  {"opacity": 0.5, "color": "lightblue"})
  view.zoomTo()                         # ajustar zoom
  view.show()                           # mostrar en Jupyter
    ''')

    print('  Esquemas de color comunes:')
    colores = [
        ('ssPyMol', 'Tipo PyMOL — colores vibrantes por elemento'),
        ('Jmol', 'Estándar Jmol — colores IUPAC clásicos'),
        ('ssJmol', 'JMol por tipo de átomo'),
        ('spectrum', 'Espectro arco iris de N-terminus a C-terminus'),
        ('chain', 'Color diferente por cadena proteica'),
    ]
    for col, desc in colores:
        print(f'  • {col:12s}: {desc}')

comparar_visualizaciones()

# Generar y mostrar moléculas
print('\n--- Generando visualizaciones ---')
for smi, nom in [('c1ccccc1', 'Benceno'), 
                  ('C1CCCCC1', 'Ciclohexano'),
                  ('CC(=O)Oc1ccccc1C(=O)O', 'Aspirina')]:
    visualizar_mol_py3dmol(smi, nom)

## 4. Flujo de Trabajo Integrado

Un flujo real de trabajo combina múltiples herramientas:

In [ ]:
def pipeline_completo(smiles, nombre, T=298.15):
    """
    Pipeline completo de análisis molecular:
    SMILES → 3D → Optimización → Propiedades → Visualización
    """
    print(f'\n{'='*60}')
    print(f'  PIPELINE: {nombre}')
    print(f'  SMILES: {smiles}')
    print(f'{'='*60}')

    # Paso 1: Conversión SMILES → Mol
    print('\n[1/5] Lectura de SMILES...')
    if not RDKIT_OK:
        print('  (sin RDKit — usando datos simulados)')
        print(f'  Fórmula molecular estimada: C_n H_m O_k')
        return

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f'  ✗ Error: SMILES inválido')
        return
    formula = rdMolDescriptors.CalcMolFormula(mol)
    print(f'  ✓ Fórmula: {formula}, {mol.GetNumAtoms()} átomos pesados')

    # Paso 2: Generación de conformeros
    print('\n[2/5] Generación de ensemble conformacional...')
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    n = AllChem.EmbedMultipleConfs(mol, numConfs=50, params=params)
    print(f'  ✓ {n} conformeros generados')

    # Paso 3: Optimización
    print('\n[3/5] Optimización MMFF94...')
    res = AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=500)
    energias = [r[1] for r in res if r[0] == 0]
    E_arr = np.array(energias)
    E_rel = E_arr - E_arr.min()
    print(f'  ✓ {len(energias)} conformeros optimizados')
    print(f'  Rango E: {E_rel.min():.2f} – {E_rel.max():.2f} kcal/mol')

    # Paso 4: Propiedades
    print('\n[4/5] Cálculo de propiedades...')
    mol_2d = Chem.RemoveHs(mol)
    mw = Descriptors.MolWt(mol_2d)
    logp = Descriptors.MolLogP(mol_2d)
    tpsa = Descriptors.TPSA(mol_2d)
    hbd = rdMolDescriptors.CalcNumHBD(mol_2d)
    hba = rdMolDescriptors.CalcNumHBA(mol_2d)

    conf = mol.GetConformer(0)
    pos = conf.GetPositions()
    Rg = np.sqrt(((pos - pos.mean(axis=0))**2).sum(axis=1).mean())

    pesos = np.exp(-E_rel / (1.987e-3 * T))
    pesos /= pesos.sum()
    pop_global = pesos[0] * 100

    print(f'  MW = {mw:.1f} Da, LogP = {logp:.2f}, TPSA = {tpsa:.1f} Å²')
    print(f'  HBD = {hbd}, HBA = {hba}')
    print(f'  Radio de giro (global mín): {Rg:.2f} Å')
    print(f'  Población del global a {T:.0f} K: {pop_global:.1f}%')

    lipinski = mw <= 500 and logp <= 5 and hbd <= 5 and hba <= 10
    print(f'  Lipinski: {"✓ Cumple" if lipinski else "✗ Viola"}')

    # Paso 5: Export
    print('\n[5/5] Exportación de resultados...')
    molblock = Chem.MolToMolBlock(mol)
    pdbblock = Chem.MolToPDBBlock(mol)
    print(f'  ✓ MOL block: {len(molblock)} caracteres')
    print(f'  ✓ PDB block: {len(pdbblock)} caracteres')

    # Visualización Py3Dmol
    if PY3DMOL_OK:
        view = py3Dmol.view(width=500, height=350)
        view.addModel(molblock, 'mol')
        view.setStyle({'stick': {}})
        view.zoomTo()
        print('  ✓ Visualización 3D:')
        view.show()

    print(f'\n{'='*60}')
    print(f'  PIPELINE COMPLETADO — {nombre}')
    print(f'{'='*60}\n')

    return {'formula': formula, 'MW': mw, 'LogP': logp, 'TPSA': tpsa,
           'Rg': Rg, 'conformeros': len(energias), 'lipinski': lipinski}

# Ejecutar pipeline para moléculas de interés
resultados_pipeline = {}
for smi, nom in [
    ('CC(=O)Oc1ccccc1C(=O)O', 'Aspirina'),
    ('CN1C=NC2=C1C(=O)N(C(=O)N2C)C', 'Cafeína'),
]:
    res = pipeline_completo(smi, nom)
    if res:
        resultados_pipeline[nom] = res

## 5. Comparación de Herramientas

In [ ]:
# Tabla comparativa de software para MM/quiminformática
import pandas as pd

datos_software = [
    ('RDKit', 'Python', 'Libre', '★★★★★',
     'Quiminformática, descriptores, conformeros, fingerprints',
     'No MD, sin dinámica avanzada'),
    ('OpenBabel', 'CLI/Python', 'Libre', '★★★★☆',
     'Conversión formatos (>110), SMILES, minimización rápida',
     'Campos de fuerza básicos'),
    ('Py3Dmol', 'Python', 'Libre', '★★★★☆',
     'Visualización 3D Jupyter, superficies, animaciones MD',
     'Solo visualización, sin cálculos'),
    ('GROMACS', 'CLI', 'Libre', '★★★★★',
     'MD biomolecular, altísimo rendimiento, GPU',
     'Curva de aprendizaje alta, preparación compleja'),
    ('Tinker', 'CLI', 'Libre', '★★★☆☆',
     'Educacional, muchos CdF, fácil de usar',
     'Menos activo, interfaces limitadas'),
    ('NAMD', 'CLI', 'Libre', '★★★★☆',
     'MD en GPU, escalable, CHARMM/AMBER',
     'Requiere VMD para preparación'),
    ('Schrodinger', 'GUI/CLI', 'Comercial', '★★★★★',
     'Suite completa, docking, MM/QMMM, ADMET',
     'Costo elevado, licencia requerida'),
    ('AMBER', 'CLI', 'Académico', '★★★★★',
     'FF biomoleculares de referencia, MD, QM/MM',
     'Licencia, preparación compleja'),
]

df_soft = pd.DataFrame(datos_software,
    columns=['Software', 'Interfaz', 'Licencia', 'Rating', 'Fortalezas', 'Limitaciones'])

print('\n📊 COMPARATIVA DE SOFTWARE PARA MECÁNICA MOLECULAR')
print('='*110)
print(f'{"Software":12s} {"Interfaz":12s} {"Licencia":12s} {"Rating":10s} {"Fortalezas":50s}')
print('-'*110)
for _, row in df_soft.iterrows():
    print(f'{row["Software"]:12s} {row["Interfaz"]:12s} {row["Licencia"]:12s} {row["Rating"]:10s} {row["Fortalezas"]:50s}')

print()
# Gráfico de cobertura funcional
software = ['RDKit', 'OpenBabel', 'Py3Dmol', 'GROMACS', 'Tinker', 'Schrodinger']
categorias = ['Quiminformática', 'Conversión formatos', 'Visualización',
             'Dinámica MD', 'Optimización MM', 'Análisis ADME']

scores = np.array([
    [5, 3, 2, 0, 4, 5],  # RDKit
    [3, 5, 1, 0, 3, 2],  # OpenBabel
    [0, 0, 5, 1, 0, 0],  # Py3Dmol
    [1, 2, 2, 5, 4, 2],  # GROMACS
    [2, 2, 1, 3, 5, 2],  # Tinker
    [5, 4, 5, 5, 5, 5],  # Schrodinger
])

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(categorias))
width = 0.13
colores_soft = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#795548', '#F44336']

for i, (sw, col) in enumerate(zip(software, colores_soft)):
    offset = (i - len(software)/2 + 0.5) * width
    ax.bar(x + offset, scores[i], width, label=sw, color=col, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(categorias, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Capacidad (0–5)', fontsize=12)
ax.set_title('Cobertura Funcional por Software', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, ncol=3)
ax.set_ylim(0, 5.5)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Ejercicios Prácticos

### Ejercicio 1 (Básico)
Usa `demostrar_rdkit` para analizar los siguientes fármacos:
- Paracetamol: `CC(=O)Nc1ccc(O)cc1`
- Ibuprofeno: `CC(C)Cc1ccc(C(C)C(=O)O)cc1`
- Diazepam: `CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21`

Compara sus descriptores ADME y explica cuál tiene mejor perfil farmacocinético.

### Ejercicio 2 (Intermedio)
Ejecuta `pipeline_completo` para la **cafeína** y el **teofilina** (`Cn1cnc2c1c(=O)[nH]c(=O)n2C`). Compara el número de conformeros dentro de 2 kcal/mol y el radio de giro promedio del ensemble.

### Ejercicio 3 (Avanzado)
Si tienes OpenBabel disponible, escribe un script que:
1. Lea una lista de SMILES
2. Genere estructuras 3D con `obabel --gen3d`
3. Optimice con `obabel --minimize --ff MMFF94`
4. Exporte a PDB para cada molécula
5. Calcule el RMSD entre la estructura OpenBabel y RDKit MMFF94

In [ ]:
# Ejercicio 1: Análisis de fármacos
farmacos_ex = [
    'CC(=O)Nc1ccc(O)cc1',  # Paracetamol
    'CC(C)Cc1ccc(C(C)C(=O)O)cc1',  # Ibuprofeno
    'CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21',  # Diazepam
]
df_ex1 = demostrar_rdkit(farmacos_ex)
# Tu código para ejercicios 2 y 3 aquí...

## 7. Referencias

1. Landrum, G. (2024). RDKit: Open-source cheminformatics. https://www.rdkit.org/
2. O'Boyle, N. M. et al. (2011). Open Babel: An Open Chemical Toolbox. *J. Cheminform.*, 3, 33.
3. Rego, N. & Bhatt, D. (2015). 3Dmol.js: molecular visualization with WebGL. *Bioinformatics*, 31(8), 1322–1324.
4. Van Der Spoel, D. et al. (2005). GROMACS: Fast, flexible, and free. *J. Comput. Chem.*, 26(16), 1701–1718.
5. Ponder, J. W. & Case, D. A. (2003). Force fields for protein simulations. *Adv. Prot. Chem.*, 66, 27–85.

---

## 📚 Recursos Adicionales

### Documentación
- [RDKit Getting Started](https://www.rdkit.org/docs/GettingStartedInPython.html)
- [OpenBabel Documentation](https://openbabel.org/docs/)
- [Py3Dmol GitHub](https://github.com/3dmol/3Dmol.js)
- [GROMACS Manual](https://manual.gromacs.org/)

### Tutoriales
- [RDKit Cookbook](https://www.rdkit.org/docs/Cookbook.html)
- [Open Babel Tutorial](http://openbabel.org/wiki/Tutorial)

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Describir las fortalezas y limitaciones del software principal de MM
- ✅ Usar RDKit para generar conformeros, calcular descriptores y exportar formatos
- ✅ Convertir formatos moleculares con OpenBabel desde la terminal o Python
- ✅ Crear visualizaciones 3D interactivas con Py3Dmol en Jupyter
- ✅ Diseñar un pipeline que integre múltiples herramientas
- ✅ Seleccionar la herramienta adecuada para cada tarea computacional

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 3.7: Software Especializado**

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_3.6-Cálculo_Propiedades-blue.svg)](06_calculo_propiedades.ipynb)
[![Siguiente](https://img.shields.io/badge/Actividad_3.8_➡️-Validación_Resultados-green.svg)](08_validacion_resultados.ipynb)

---

📚 **[Volver al Módulo 3](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G - 2026*

</div>